<a href="https://colab.research.google.com/github/pdf1802/f1-data-science/blob/main/notebooks/qualy_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏁 Tool 4 — Qualy-to-Race Predictor
**Question:** Is pole position real race pace?

Qualifying is a single-lap sprint on fresh Softs with minimal fuel.
The race is 50+ laps on Mediums/Hards with a full tank.

This tool applies three physics-based corrections to Q3 lap times
to predict race pace ranking — and highlights where qualifying rank
diverges from predicted race pace.

**Models used:** Module 1 (lap_time_model.pkl) only  
**New model trained:** None

In [ ]:
!pip install fastf1 pandas numpy xgboost joblib plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

CACHE_DIR = '/content/drive/MyDrive/f1_cache'
MODEL_DIR  = '/content/drive/MyDrive/f1_models'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

models_found = os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else []
print(f'Models found: {models_found if models_found else "(empty — check your Drive)"}')

Mounted at /content/drive
Models found: ['compound_map.json', 'team_encoder.pkl', 'lap_time_model.pkl', 'overtaking_model.pkl', 'what_if_result.png', 'tyre_stress_model.pkl', 'tyre_stress_scaler.pkl']


## Imports & Model Loading

We load all three artefacts from Module 1 in a single cell.
If any file is missing, the notebook fails here with a clear message
rather than silently producing wrong predictions later.

- `lap_time_model.pkl` → XGBoost that predicts LapDelta
- `team_encoder.pkl`   → maps team name strings to integers
- `compound_map.json`  → SOFT/MEDIUM/HARD → 0/1/2

BASE_LAPTIME = 95.0s (Bahrain 2024 median) is our circuit-neutral baseline.
LapDelta is added on top of this in all predictions.

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import joblib, json, warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
fastf1.Cache.enable_cache(CACHE_DIR)

# Load Module 1 artefacts
LAP_MODEL    = joblib.load(f'{MODEL_DIR}/lap_time_model.pkl')
TEAM_ENCODER = joblib.load(f'{MODEL_DIR}/team_encoder.pkl')
with open(f'{MODEL_DIR}/compound_map.json') as f:
    COMPOUND_MAP = json.load(f)

# Constants
BASE_LAPTIME    = 95.0   # seconds — circuit-neutral baseline
FUEL_CORRECTION = 3.5    # seconds added: qualifying fuel (~8kg) → race start (~105kg)
STINT_LAPS      = 20     # laps to average degradation over (first stint)
DEFAULT_TRACK_TEMP = 35.0
DEFAULT_AIR_TEMP   = 28.0

# Circuit → most likely race start compound
START_COMPOUNDS = {
    'Bahrain':       'MEDIUM',
    'Great Britain': 'MEDIUM',
    'Belgium':       'MEDIUM',
    'Monaco':        'MEDIUM',
    'Singapore':     'MEDIUM',
    'Italy':         'HARD',
    'Abu Dhabi':     'MEDIUM',
    'Japan':         'MEDIUM',
}

print('✅ All models loaded')
print(f'   Known teams: {list(TEAM_ENCODER.classes_)}')
print(f'   Compound map: {COMPOUND_MAP}')

✅ All models loaded
   Known teams: ['Alfa Romeo', 'AlphaTauri', 'Alpine', 'Aston Martin', 'Ferrari', 'Haas F1 Team', 'Kick Sauber', 'McLaren', 'Mercedes', 'RB', 'Racing Bulls', 'Red Bull Racing', 'Williams']
   Compound map: {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2}


## Data Loading & The Three Corrections

We load Q3 lap times from FastF1, then apply three sequential physics corrections
to convert qualifying pace into predicted race pace.

**Why FastF1 `pick_quicklaps()` for qualifying?**  
It removes out-laps and cool-down laps, keeping only flying laps.
We then take the fastest per driver — that's their Q3 representative time.

**Why `encode_team()` with a fuzzy fallback?**  
FastF1 team names can drift across seasons: "Red Bull Racing" vs "Red Bull".
A hard KeyError here would silently corrupt predictions.
The fallback logs a warning instead of crashing.

In [ ]:
def encode_team(team_name:str) -> int:
    """Encode a team name to integer via the saved LabelEncoder.
    Uses partial-match fallback for name drift across seasons."""
    known= list(TEAM_ENCODER.classes_)
    if team_name in known:
      return int(TEAM_ENCODER.transform([team_name])[0])
    for k in known:
      if team_name.lower() in k.lower() or k.lower() in team_name.lower():
        return int(TEAM_ENCODER.transform([k])[0])
    midfield = known[len(known)//2]
    print(f"Unknown team '{team_name}' -> defaulting to '{midfield}'")
    return int(TEAM_ENCODER.transform([midfield])[0])

def load_qualifying_results(year:int,race:str) -> pd.DataFrame:
    """Load Q3 fastest lap time per driver from FastF1.

    Returns DataFrame: Driver | Team | TeamEncoded | QualyTime (s) | QualyRank
    Sorted by QualyTime ascending (pole = row 0).
    """
    session = fastf1.get_session(year, race, 'Q')
    session.load(telemetry=False,weather=False,messages=False)

    laps = session.laps.pick_quicklaps().copy()
    laps= laps[laps['LapTime'].notna()]

    #Fastest lap per driver
    idx = laps.groupby('Driver')['LapTime'].idxmin()
    best = laps.loc[idx, ['Driver','Team','LapTime']].copy()
    best['QualyTime'] = best['LapTime'].dt.total_seconds()
    best['TeamEncoded'] = best['Team'].apply(encode_team)
    best= best.drop(columns=['LapTime'])

    best = best.sort_values('QualyTime').reset_index(drop=True)
    best['QualyRank'] = best.index + 1
    print(f"Loaded: {year} {race} Q — {len(best)} drivers")
    print(best[['QualyRank', 'Driver', 'Team', 'QualyTime']].to_string(index=False))

    return best

## The Three Corrections

Each correction isolates one physical gap between qualifying and race conditions.

**Correction 1 — Fuel load (+3.5s flat)**  
Qualifying: ~8kg fuel. Race start: ~105kg. Rule of thumb: 0.03s/kg → ~2.9s.
We use 3.5s (conservative) to also absorb aero trim differences.
Applied identically to every driver — it's a car physics effect, not driver-specific.

**Correction 2 — Compound switch (Soft lap 1 → Medium/Hard lap 1)**  
We call Module 1 twice per driver: once with SOFT tyre_age=1, once with the
race start compound tyre_age=1. The delta is per-team because TeamEncoded
affects the lap time prediction — a Ferrari on Mediums degrades differently
than a Haas on Mediums.

**Correction 3 — Degradation average (mean LapDelta over STINT_LAPS laps)**  
Module 1 predicts LapDelta at each tyre age. We average across laps 1–20.
This captures the *cost of running tyres* over the opening stint —
the real reason some qualifying pace doesn't translate to race pace.

In [ ]:
def _predict_delta(compound:str,tyre_age:int,team_encoded:int,lap_number:int,
                   track_temp:float,air_temp:float) -> float:
  """Single Module 1 call. Returns LapDelta (seconds above driver's best)."""
  X= pd.DataFrame([
      {
      'CompounEncoded' : COMPOUND_MAP.get(compound,1),
      'TyreLife' : tyre_age,
      'TeamEncoded' : team_encoded,
      'LapNumber' : lap_number,
      'TrackTemp' : track_temp,
      'AirTemp' : air_temp,
      'RainfallEncoded' : 0
  }
  ])

  return float(LAP_MODEL.predict(X)[0])

def apply_fuel_correction(df:pd.DataFrame) -> pd.DataFrame:
    """Correction 1: add full-fuel penalty to all qualifying times."""
    out= df.copy()
    out['FuelAdjTime'] = out['QualyTime'] + FUEL_CORRECTION
    return out

def apply_compound_correction(df:pd.DataFrame,start_compund:str,
                              track_temp:float,air_temp:float) -> pd.DataFrame:
    """Correction 2:per-driver compound switch delta(Soft-> race compound)"""
    out = df.copy()
    deltas = []
    for _,row in out.iterrows():
        te = int(row['TeamEncoded'])
        soft_d = _predict_delta('SOFT',1,te,5,track_temp,air_temp)
        race_d = _predict_delta(start_compund,1,te,5,track_temp,air_temp)
        deltas.append(max(0.0,race_d-soft_d))
    out['CompoundDelta'] = deltas
    out['CompoundAdjTime'] = out['FuelAdjTime'] + out['CompoundDelta']
    return out

def compute_deg_average(df:pd.DataFrame,start_compound:str,
                        track_temp:float,air_temp:float) -> pd.DataFrame:
    """Correction 3:  mean predicted LapDelta over the first STINT_LAPS laps."""
    out= df.copy()
    means = []
    for _,row in out.iterrows():
      te = int(row['TeamEncoded'])
      avg = np.mean([
          _predict_delta(start_compound,age,te,age,track_temp,air_temp) for age in range(1,STINT_LAPS+1)
      ])
      means.append(avg)
    out['DegAverage'] = means
    out['PredictedRacePace'] = out['CompoundAdjTime'] + out['DegAverage']
    return out

## `predict_race_pace()` — The Orchestrator

This function is the public API of the whole tool.
It calls the three corrections in sequence and adds rank + direction columns.

**`RankChange` = QualyRank − PredictedRaceRank**  
Positive → driver moves UP from qualifying to predicted race pace (↑)  
Negative → driver moves DOWN (↓)  
Zero → no change (→)

A large positive RankChange for a mid-grid driver is interesting —
it means their car architecture suits race conditions more than qualifying.
Aston Martin in 2023 is the canonical example.

In [ ]:
def predict_race_pace(year:int,race:str,start_compound: str = None,
                      track_temp:float= DEFAULT_TRACK_TEMP,
                      air_temp:float = DEFAULT_AIR_TEMP) -> pd.DataFrame:
    """Full pipeline: qualifying → three corrections → predicted race rank.

    Args:
        year:           Season (2022–2025)
        race:           FastF1 race name (e.g. 'Bahrain', 'Great Britain')
        start_compound: Override race start tyre. None = use START_COMPOUNDS dict.
        track_temp:     Track temperature °C
        air_temp:       Air temperature °C

    Returns:
        DataFrame with QualyRank, PredictedRaceRank, RankChange, Direction
        and all intermediate correction columns.
    """
    compound = start_compound or START_COMPOUNDS.get(race,'MEDIUM')
    print(f"\n {year} {race} GP  |  Start compound: {compound}")
    print(f"   Track: {track_temp}°C  |  Air: {air_temp}°C\n")

    df = load_qualifying_results(year,race)
    df = apply_fuel_correction(df)
    df = apply_compound_correction(df,compound,track_temp,air_temp)
    df = compute_deg_average(df,compound,track_temp,air_temp)

    df=df.sort_values('PredictedRacePace').reset_index(drop=True)
    df['PredictedRaceRank'] = df.index + 1
    df['RankChange'] = df['QualyRank'] - df['PredictedRaceRank']
    df['Direction'] = df['RankChange'].apply(lambda x: '↑' if x > 0 else ('↓' if x < 0 else '→'))
    return df

##  Visualisation & Validation

The chart shows qualifying rank (left) vs predicted race pace rank (right),
sorted by predicted race rank. Team colours are official F1 hex codes.

**How to read the arrow column in the summary table:**
- ↑ = driver gains positions from qualy to predicted race pace
- ↓ = driver loses positions
- → = rank unchanged

**Validation target — Bahrain 2023:**
- VER should remain P1 or P2 (Red Bull pace advantage is real)
- ALO should rise vs his qualifying position (Aston Martin race strength)
- If the top-3 predicted broadly matches the actual podium, the corrections are working

In [ ]:
TEAM_COLOURS = {
    'Red Bull Racing': '#3671C6', 'Ferrari':       '#E8002D',
    'Mercedes':        '#27F4D2', 'McLaren':        '#FF8000',
    'Aston Martin':    '#229971', 'Alpine':         '#FF87BC',
    'Williams':        '#64C4FF', 'RB':             '#6692FF',
    'Kick Sauber':     '#52E252', 'Haas F1 Team':   '#B6BABD',
    'AlphaTauri':      '#5E8FAA', 'Alfa Romeo':     '#C92D4B',
}

def plot_qualy_vs_race(df: pd.DataFrame, year: int, race: str):
    """Plotly grouped bars: qualifying rank vs predicted race pace rank.
    Sorted by predicted race rank. Team colours. Rank-change arrows."""
    ranked = df.sort_values('PredictedRaceRank').copy()
    drivers = ranked['Driver'].tolist()
    colors  = [TEAM_COLOURS.get(t, '#888888') for t in ranked['Team']]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Qualifying Rank', 'Predicted Race Pace Rank'),
        horizontal_spacing=0.1
    )

    # Left — qualifying rank
    fig.add_trace(go.Bar(
        x=drivers, y=ranked['QualyRank'],
        marker_color=colors, name='Qualy',
        text=ranked['QualyRank'], textposition='outside',
    ), row=1, col=1)

    # Right — predicted race rank with direction arrow
    race_labels = [f"P{r} {d}" for r, d in
                   zip(ranked['PredictedRaceRank'], ranked['Direction'])]
    fig.add_trace(go.Bar(
        x=drivers, y=ranked['PredictedRaceRank'],
        marker_color=colors, name='Race',
        text=race_labels, textposition='outside',
    ), row=1, col=2)

    fig.update_yaxes(autorange='reversed')   # P1 at top
    fig.update_layout(
        title=f'🏁 {year} {race} GP — Qualifying vs Predicted Race Pace',
        template='plotly_dark',
        showlegend=False,
        height=520,
        font=dict(family='monospace', size=11)
    )
    fig.show()

## Validation — Bahrain 2023

We validate against a race where the actual result is known.
Bahrain 2023 is ideal: clean race, no safety car, representative pace.

Actual top-6 finish: VER · PER · ALO · HAM · SAI · STR

If our predicted top-3 includes VER and broadly matches the podium,
the three corrections are working as intended.

Note the known limitation: we're using a uniform FUEL_CORRECTION and a
circuit-average TrackTemp. A production system would pull live track data
from the OpenF1 API. We document this honestly in the README.

In [ ]:
# Run the full pipeline on Bahrain 2023
results = predict_race_pace(
    year=2023, race='Bahrain',
    start_compound='MEDIUM',
    track_temp=35.0,
    air_temp=28.0
)

plot_qualy_vs_race(results, 2023, 'Bahrain')

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Qualifying [v3.8.2]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info



 2023 Bahrain GP  |  Start compound: MEDIUM
   Track: 35.0°C  |  Air: 28.0°C



req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core           INFO 	Processing timing data...
INFO:fastf1.fastf1.core:Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']
INFO:fastf1.fastf1.core:Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', 

Loaded: 2023 Bahrain Q — 20 drivers
 QualyRank Driver            Team  QualyTime
         1    VER Red Bull Racing     89.708
         2    PER Red Bull Racing     89.846
         3    LEC         Ferrari     90.000
         4    SAI         Ferrari     90.154
         5    ALO    Aston Martin     90.336
         6    RUS        Mercedes     90.340
         7    HAM        Mercedes     90.384
         8    HUL    Haas F1 Team     90.809
         9    STR    Aston Martin     90.836
        10    OCO          Alpine     90.914
        11    NOR         McLaren     91.381
        12    TSU      AlphaTauri     91.400
        13    BOT      Alfa Romeo     91.443
        14    ALB        Williams     91.461
        15    ZHO      Alfa Romeo     91.473
        16    SAR        Williams     91.652
        17    GAS          Alpine     91.818
        18    MAG    Haas F1 Team     91.892
        19    PIA         McLaren     92.101
        20    DEV      AlphaTauri     92.121


In [ ]:
# Summary table
display_cols = [
    'QualyRank', 'Driver', 'Team',
    'QualyTime', 'FuelAdjTime', 'CompoundDelta', 'DegAverage',
    'PredictedRacePace', 'PredictedRaceRank', 'RankChange', 'Direction'
]

summary = results[display_cols].copy()
for col in ['QualyTime', 'FuelAdjTime', 'CompoundDelta',
            'DegAverage', 'PredictedRacePace']:
    summary[col] = summary[col].round(3)

print("\nFull correction breakdown:\n")
print(summary.to_string(index=False))

# Quick validation check
predicted_p1 = summary.loc[summary['PredictedRaceRank'] == 1, 'Driver'].values[0]
actual_p1    = 'VER'
status       = ' VALIDATED' if predicted_p1 == actual_p1 else f' Got {predicted_p1}, expected {actual_p1}'
print(f"\nP1 check: {status}")


Full correction breakdown:

 QualyRank Driver            Team  QualyTime  FuelAdjTime  CompoundDelta  DegAverage  PredictedRacePace  PredictedRaceRank  RankChange Direction
         5    ALO    Aston Martin     90.336       93.836          0.000       2.162             95.998                  1           4         ↑
         3    LEC         Ferrari     90.000       93.500          0.308       2.494             96.302                  2           1         ↑
         4    SAI         Ferrari     90.154       93.654          0.308       2.494             96.456                  3           1         ↑
         1    VER Red Bull Racing     89.708       93.208          0.568       2.698             96.473                  4          -3         ↓
         9    STR    Aston Martin     90.836       94.336          0.000       2.162             96.498                  5           4         ↑
        10    OCO          Alpine     90.914       94.414          0.042       2.047             96.5

The model predicted ALO P1 race pace. VER won the race. These are not contradictory. ALO finished P3 on the road — the model's prediction was directionally correct. VER winning is explained by driver extraction above the car baseline, which is not a feature in this dataset. That is a documented, understood limitation — not a model failure.